In [ ]:
pip install pyserial

In [ ]:
!pip install serial

In [ ]:
#Arduino Data Acquisition and CSV Export
#This notebook reads raw vibration sensor data sent by the Arduino through
#Serial USB, converts the ADC values into voltage, and stores the acquired
#signal in a CSV file for later analysis.

import serial
import time
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
import matplotlib.pyplot as plt

#1. Acquisition parameters
serial_port = "COM3"      
baudrate = 9600           # Match the Arduino Serial baudrate
duration_s = 0.3          # Acquisition time in seconds
Vref = 3.3                # ADC reference voltage of the sensor module
ADC_max = 4095            # Maximum value of the 12-bit ADC

# 2. Arduino data acquisition
time_us = []              # Lists used to store time and raw sensor values
raw_values = []

ser = serial.Serial(puerto_serial, baudrate, timeout=0.01)  # Open Serial connection with the Arduino
time.sleep(2)                                               # Wait for the Serial connection to stabilise


start_time = time.time()                            # Acquisition start time
while time.time() - start_time < duration_s:        # Read data until the acquisition time has elapsed
    line = ser.readline().decode("utf-8").strip()   # Read one line received from the Arduino
    if not line:                                    # Skip empty lines
        continue
    try:
        timestamp_us, raw_value = line.split(",")      # Split each line into timestamp and raw sensor value
        time_us.append(float(timestamp_us))
        raw_values.append(int(raw_value))              # Store the raw sensor measurement
    except: 
        continue
        
ser.close()

# 3. Signal conversion
time_us = np.array(time_us)   # Convert lists into arrays
raw_values = np.array(raw_values)

time_us = time_us - time_us[0]

t = time_us / 1000000                # Convert time from microseconds to seconds
v = raw_values * Vref / ADC_max      # Convert ADC values into voltage
v = v - np.mean(v)                   # Remove the DC offset from the signal


dt_array = np.diff(t)              # Sampling frequency verification
dt = np.mean(dt_array)
fs = 1 / dt

print(f"Sampling frequency ≈ {fs:.2f} Hz")
print(f"Number of samples acquired = {len(v)}")
print(f"Raw min/max = {raw_values.min()} / {raw_values.max()}")
print(f"Voltage AC min/max = {v.min():.4f} / {v.max():.4f} V")


# 4. Export data to a csv file
output_file = "DEF3_SM_3.3V_muestra1.csv"    # Change for each measurement (SM = Small Motor, BM = Big Motor)

min_len = min(len(t), len(raw_values), len(v))
t = t[:min_len]
raw_values = raw_values[:min_len]
v = v[:min_len]

data = pd.DataFrame({
    "time_s": t,
    "raw": raw_values,
    "voltage_V": v
})

data.to_csv(output_file, index=False, sep=";", decimal=",")

# 5. Signal visualisation
plt.figure(figsize=(10,4))
plt.plot(t, v)
plt.xlabel("Time (s)")
plt.ylabel("Voltage AC (V)")
plt.title("Acquired vibration sensor signal")
plt.grid(True)
plt.show()
 